# Notebook 02 -- Exploratory Data Analysis and Statistical Testing

## Purpose of EDA

EDA is not about making pretty plots. **Every visualization and test leads to a concrete
decision** about feature engineering or modeling strategy.

We are acting as a data scientist at a ride-hailing company. Key questions:

1. **Is `eta_seconds` well-behaved?** Skewed targets may need log-transformation.
2. **Which features correlate with ETA?** Informs feature prioritization.
3. **Do temporal patterns exist?** Rush hour, weekends -- if significant, they become features.
4. **Does operator (Uber vs Lyft) matter?** Statistically test before including as feature.
5. **Which columns leak future info?** `trip_time` is post-trip -- cannot be a feature.

By the end of this notebook we have a complete **EDA-to-Feature-Engineering decision log**
that justifies every choice in Notebook 03.

**Input:** `data/interim/sample_filtered.parquet`  
**Output:** Insights + saved plots in `reports/figures/`


In [ ]:
# Cell 2: Imports and data load
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))
import config

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = config.RANDOM_STATE
FIGURES_DIR  = Path('..') / config.REPORTS_PATH
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path('..') / config.INTERIM_DATA_PATH / config.INTERIM_FILTERED_NAME
df = pd.read_parquet(DATA_PATH)

for col in ['request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime']:
    if col in df.columns and df[col].dtype == object:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'eta_seconds range: [{df["eta_seconds"].min():.0f}s, {df["eta_seconds"].max():.0f}s]')
df.head(3)


## 2.3  Descriptive Statistics -- What Each Metric Tells Us

| Statistic | What it tells us | Decision implication |
|---|---|---|
| **Mean vs Median** | If mean >> median: right skew | Consider log-transform of target |
| **Std** | Spread of values | High std = hard prediction problem |
| **Skewness > 1** | Strong right tail | Log-transform will help linear models |
| **Kurtosis > 3** | Heavy tails, more extreme values | Robust loss function (MAE over MSE) |
| **IQR** | Middle 50%, robust to outliers | Use for outlier fence computation |

We compute these for `eta_seconds`, `trip_miles`, `trip_time`, and fare columns.
Note: `trip_time`, `base_passenger_fare`, and `driver_pay` are post-trip columns
(they CANNOT be used as features) but we analyze them here for domain understanding.


In [ ]:
# Cell 4: Descriptive statistics
ANALYSIS_COLS = [c for c in
    ['eta_seconds', 'trip_miles', 'trip_time', 'base_passenger_fare', 'driver_pay']
    if c in df.columns]

stats_df = df[ANALYSIS_COLS].describe().T
stats_df['skewness'] = df[ANALYSIS_COLS].skew()
stats_df['kurtosis'] = df[ANALYSIS_COLS].kurtosis()  # excess kurtosis
stats_df['IQR']      = df[ANALYSIS_COLS].quantile(0.75) - df[ANALYSIS_COLS].quantile(0.25)
stats_df['median']   = df[ANALYSIS_COLS].median()

col_order = ['count','mean','median','std','IQR','min','25%','75%','max','skewness','kurtosis']
col_order = [c for c in col_order if c in stats_df.columns]

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
display(stats_df[col_order])
pd.reset_option('display.float_format')


## 2.5  Interpreting Descriptive Stats

**For `eta_seconds` (our target):**
- If **mean > median**: right-skewed -- most pickups are fast, a long tail of slow ones
- If **skewness > 1**: log-transforming the target will improve linear model performance
- High **kurtosis**: heavier tails than Gaussian -- more extreme values, favor MAE over RMSE

**For `trip_miles`:**
- Right-skewed: most trips are short, airport trips create a long right tail
- Outlier-prone: IQR fencing will be applied in feature engineering

**For `base_passenger_fare` and `driver_pay`:**
- Informational only -- these are POST-TRIP and MUST NOT be used as features
- We note their distributions for business context

**Decision:** If `eta_seconds` skewness > 1 --> apply `log1p` transform to target in Notebook 03.


In [ ]:
# Cell 6: Target distribution -- raw vs log-transformed
# log1p is preferred over log because log1p(0) = 0 while log(0) = -infinity.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
sns.histplot(df['eta_seconds'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(df['eta_seconds'].mean(),   color='red',   ls='--',
                label=f'Mean: {df["eta_seconds"].mean():.0f}s')
axes[0].axvline(df['eta_seconds'].median(), color='green', ls='--',
                label=f'Median: {df["eta_seconds"].median():.0f}s')
axes[0].set_title('Distribution of eta_seconds (Raw)')
axes[0].set_xlabel('ETA (seconds)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Log-transformed
log_eta = np.log1p(df['eta_seconds'])
sns.histplot(log_eta, bins=60, kde=True, ax=axes[1], color='coral')
axes[1].axvline(log_eta.mean(),   color='red',   ls='--', label=f'Mean: {log_eta.mean():.2f}')
axes[1].axvline(log_eta.median(), color='green', ls='--', label=f'Median: {log_eta.median():.2f}')
axes[1].set_title('Distribution of log1p(eta_seconds)')
axes[1].set_xlabel('log1p(ETA seconds)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Raw vs Log-Transformed Target Distribution', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_eta_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Raw skewness:  {df["eta_seconds"].skew():.3f}')
print(f'Log skewness:  {log_eta.skew():.3f}')
print('Interpretation: log1p significantly reduces skewness -- confirms log-transform decision.')


## 2.7  Outlier Detection -- IQR vs Z-Score

Two standard methods for outlier detection:

**IQR Method**  
```
Lower bound = Q1 - 1.5 x IQR
Upper bound = Q3 + 1.5 x IQR
```
- Robust to any distribution shape, including heavily skewed
- Not influenced by the outliers themselves (Q1, Q3 are resistant statistics)

**Z-Score Method**
```
Outlier if |z| = |(x - mean) / std| > 3
```
- Assumes normality -- invalid for our skewed distributions
- Mean and std are themselves distorted by outliers (masking problem)

**Our choice: IQR method** -- our distributions are non-normal (confirmed in Cell 6).
This is the statistically principled choice when normality cannot be assumed.


In [ ]:
# Cell 8: IQR outlier detection + boxplots

def iqr_report(series, label):
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((series < lower) | (series > upper)).sum()
    pct   = 100 * n_out / len(series)
    print(f'{label}:  Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}')
    print(f'  Bounds: [{lower:.1f}, {upper:.1f}]  Outliers: {n_out:,} ({pct:.2f}%)')
    return lower, upper

print('--- IQR Outlier Analysis ---')
iqr_report(df['eta_seconds'], 'eta_seconds')
print()
iqr_report(df['trip_miles'],  'trip_miles')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, lbl in zip(axes,
    ['eta_seconds', 'trip_miles'],
    ['ETA (seconds)', 'Trip Miles']):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'Boxplot: {col}')
    ax.set_ylabel(lbl)
    ax.set_xticks([])

plt.suptitle('IQR Outlier Detection (whiskers = 1.5 x IQR)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Points beyond whiskers are IQR outliers -- will be capped during feature engineering.')


## 2.9  Hypothesis Test: Uber vs Lyft ETA

**Business question:** Do Uber and Lyft have statistically different pickup ETAs?
If yes, the operator identifier carries predictive signal and must be a feature.

**Why Mann-Whitney U and NOT a t-test?**  
The t-test requires normality within each group. We confirmed in Cell 6 that `eta_seconds`
is right-skewed -- the normality assumption is violated. Using a t-test on non-normal data
risks elevated Type I error rates (false positives).

Mann-Whitney U is a non-parametric rank-based test:
- Makes **no distributional assumptions**
- Tests whether one group's values tend to be larger than the other's
- Has nearly equal statistical power to the t-test for large samples

**Decision rule:** p < 0.05 = significant difference = `hvfhs_license_num` is a feature.


In [ ]:
# Cell 10: Mann-Whitney U -- Uber vs Lyft ETA

df['operator'] = df['hvfhs_license_num'].map({'HV0003': 'Uber', 'HV0005': 'Lyft'}).fillna('Other')

uber_eta = df.loc[df['operator'] == 'Uber', 'eta_seconds'].dropna()
lyft_eta = df.loc[df['operator'] == 'Lyft', 'eta_seconds'].dropna()

print(f'Uber: {len(uber_eta):,} trips  | Median ETA: {uber_eta.median():.1f}s')
print(f'Lyft: {len(lyft_eta):,} trips  | Median ETA: {lyft_eta.median():.1f}s')

U_stat, p_value = stats.mannwhitneyu(uber_eta, lyft_eta, alternative='two-sided')
print(f'\nMann-Whitney U = {U_stat:.0f}  |  p-value = {p_value:.6f}')

if p_value < 0.05:
    print('RESULT: Significant difference (p < 0.05) -- operator IS a feature.')
else:
    print('RESULT: No significant difference -- operator may not add predictive value.')

fig, ax = plt.subplots(figsize=(8, 6))
plot_df = df[df['operator'].isin(['Uber', 'Lyft'])]
sns.violinplot(data=plot_df, x='operator', y='eta_seconds',
               palette=['#1DB954', '#FF00BF'], inner='quartile', ax=ax)
ax.set_title(f'ETA by Operator  (Mann-Whitney p = {p_value:.4f})')
ax.set_xlabel('Operator')
ax.set_ylabel('ETA (seconds)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_uber_lyft_violin.png', dpi=120, bbox_inches='tight')
plt.show()


## 2.11  Central Limit Theorem (CLT) Demonstration

**What is the CLT?**

> The sampling distribution of the sample mean approaches a **normal distribution**
> as sample size increases, **regardless of the shape of the underlying population**.

Formally: if X_1 ... X_n are i.i.d. from any distribution with mean mu and variance sigma^2:

    X_bar_n -> Normal(mu, sigma^2 / n)  as n -> infinity

**Why this matters for ML:**
Our statistical tests (Mann-Whitney, confidence intervals) technically assume that
test statistics are normally distributed. The CLT guarantees this for any distribution
as long as our sample is large enough (n >= 30 is the common rule of thumb).

This means even though raw `eta_seconds` is heavily skewed, we can still apply
standard statistical tests on sample means without worrying about the raw distribution's shape.

**We will demonstrate this empirically:** 1000 samples of size 50 from skewed `eta_seconds`
should produce approximately normal sample means.


In [ ]:
# Cell 12: CLT demonstration
np.random.seed(RANDOM_STATE)

N_SAMPLES   = 1000
SAMPLE_SIZE = 50
eta_arr     = df['eta_seconds'].dropna().values

sample_means = np.array([
    np.random.choice(eta_arr, size=SAMPLE_SIZE, replace=True).mean()
    for _ in range(N_SAMPLES)
])

mu, sigma = sample_means.mean(), sample_means.std()
x_norm    = np.linspace(sample_means.min(), sample_means.max(), 200)
y_norm    = stats.norm.pdf(x_norm, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(eta_arr[:5000], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Population: eta_seconds (Skewed)')
axes[0].set_xlabel('ETA (seconds)')

sns.histplot(sample_means, bins=40, stat='density', ax=axes[1],
             color='coral', label='Sample means')
axes[1].plot(x_norm, y_norm, 'k-', lw=2,
             label=f'Normal fit (mean={mu:.1f}, std={sigma:.1f})')
axes[1].set_title(f'Sampling Distribution of Mean (n={SAMPLE_SIZE}, {N_SAMPLES} samples)')
axes[1].set_xlabel('Sample Mean ETA (seconds)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('CLT: Skewed Population -> Normal Sampling Distribution', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_clt.png', dpi=120, bbox_inches='tight')
plt.show()

_, sw_p = stats.shapiro(sample_means[:500])
print(f'Shapiro-Wilk on sample means: p = {sw_p:.4f}')
print('CLT confirmed -- sample means are approximately normal.' if sw_p > 0.05
      else 'Note: mild non-normality; CLT approximation improves with larger n.')


## 2.13  Correlation -- Pearson vs Spearman

| Method | Measures | Assumes | Robust to outliers? |
|---|---|---|---|
| **Pearson** | Linear relationships | Bivariate normality | No |
| **Spearman** | Monotonic relationships | None | Yes |

**Our choice: Spearman** because:
1. `eta_seconds` is right-skewed -- violates Pearson's normality assumption
2. Relationships are likely monotonic but non-linear (doubling distance != doubling ETA)
3. We have confirmed outliers in both `eta_seconds` and `trip_miles`

Spearman correlation of +0.5 means: as `trip_miles` increases, `eta_seconds` tends
to increase monotonically -- not necessarily proportionally.

Correlating with `trip_time` and `base_passenger_fare` is informational only --
these are post-trip columns that CANNOT be features. But high correlation with eta
validates our target construction.


In [ ]:
# Cell 14: Spearman correlation heatmap
NUM_COLS = [c for c in
    ['eta_seconds','trip_miles','trip_time','base_passenger_fare','driver_pay']
    if c in df.columns]

spearman_corr = df[NUM_COLS].corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))
sns.heatmap(spearman_corr, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 11})
ax.set_title('Spearman Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_spearman_corr.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nCorrelation with eta_seconds:')
eta_corrs = spearman_corr['eta_seconds'].drop('eta_seconds').sort_values(key=abs, ascending=False)
LEAKY = ['trip_time','base_passenger_fare','driver_pay','trip_miles']
for col, val in eta_corrs.items():
    flag = '  <-- POST-TRIP (LEAKY -- cannot use as feature)' if col in LEAKY else ''
    print(f'  {col:<25} {val:+.3f}{flag}')


## 2.15  Rush Hour Pattern

**Business hypothesis:** Pickup ETAs are longer during rush hours because:
1. Traffic congestion slows drivers traveling to pickup
2. Rider demand spikes -- drivers dispatched from farther away
3. Surge pricing attracts drivers but with a lag, creating a temporary supply gap

Rush hours defined as:
- **Morning:** 07:00-09:59
- **Evening:** 17:00-19:59

If average ETA is measurably higher in these windows, we create the `is_rush_hour`
binary feature and `pickup_hour` numeric feature.


In [ ]:
# Cell 16: ETA by hour -- rush hour pattern
df['pickup_hour'] = df['request_datetime'].dt.hour
hourly = df.groupby('pickup_hour')['eta_seconds'].agg(['mean','median','count']).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(hourly['pickup_hour'], hourly['mean'],   'o-', color='steelblue', lw=2, label='Mean ETA')
ax.plot(hourly['pickup_hour'], hourly['median'], 's--', color='coral',    lw=2, label='Median ETA')
ax.axvspan(7, 9.99,   alpha=0.15, color='red',    label='Morning rush (7-9am)')
ax.axvspan(17, 19.99, alpha=0.15, color='orange', label='Evening rush (5-7pm)')
ax.set_title('Average Pickup ETA by Hour of Day', fontsize=14)
ax.set_xlabel('Hour of Day (0 = Midnight)')
ax.set_ylabel('ETA (seconds)')
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_eta_by_hour.png', dpi=120, bbox_inches='tight')
plt.show()

rush    = df[df['pickup_hour'].isin([7,8,9,17,18,19])]['eta_seconds'].mean()
offpeak = df[~df['pickup_hour'].isin([7,8,9,17,18,19])]['eta_seconds'].mean()
print(f'Mean ETA rush hours:    {rush:.1f}s')
print(f'Mean ETA off-peak:      {offpeak:.1f}s')
print(f'Rush hour premium:      {rush - offpeak:+.1f}s  ({100*(rush/offpeak-1):+.1f}%)')
print('Decision: pickup_hour (numeric) and is_rush_hour (binary) will be features.')


## 2.17  Weekend vs Weekday Hypothesis Test

**Business hypothesis:** Weekend trips have different ETA distributions because:
- Weekend riders travel to entertainment venues with less predictable demand
- Driver supply differs (surge attracts more drivers at night, fewer in daytime)
- Traffic patterns shift (lower morning congestion, higher late-night congestion)

**Test:** Mann-Whitney U (same justification as Cell 10 -- non-normal data).  
**Decision rule:**
- p < 0.05 -> `is_weekend` is a useful binary feature
- p >= 0.05 -> `is_weekend` may be redundant given `pickup_hour` already captures this


In [ ]:
# Cell 18: Weekend vs weekday Mann-Whitney U
df['pickup_day_of_week'] = df['request_datetime'].dt.dayofweek
df['is_weekend']         = df['pickup_day_of_week'].isin([5, 6]).astype(int)
df['day_type']           = df['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

weekend = df[df['is_weekend']==1]['eta_seconds'].dropna()
weekday = df[df['is_weekend']==0]['eta_seconds'].dropna()

print(f'Weekend: {len(weekend):,} trips | Median ETA: {weekend.median():.1f}s')
print(f'Weekday: {len(weekday):,} trips | Median ETA: {weekday.median():.1f}s')

U_stat, p_value = stats.mannwhitneyu(weekend, weekday, alternative='two-sided')
print(f'\nMann-Whitney U = {U_stat:.0f}  |  p-value = {p_value:.6f}')
print('is_weekend WILL be a feature.' if p_value < 0.05
      else 'is_weekend may be redundant -- reassess after model selection.')

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='day_type', y='eta_seconds',
            palette=['#4C9BE8','#F4A261'], showfliers=False, ax=ax)
ax.set_title(f'ETA by Day Type  (Mann-Whitney p = {p_value:.4f})')
ax.set_xlabel('Day Type')
ax.set_ylabel('ETA (seconds)')
ax.text(0.5, 0.97, '(Outliers hidden for visual clarity)',
        transform=ax.transAxes, ha='center', fontsize=9, color='gray', va='top')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_weekend_eta.png', dpi=120, bbox_inches='tight')
plt.show()


## 2.19  Zone-Level ETA Analysis

**Hypothesis:** Some pickup zones are consistently slower due to:
- **Urban density:** Midtown Manhattan has high demand AND high congestion
- **Driver supply gaps:** Outer boroughs may have fewer drivers per request
- **Airport zones:** JFK, LGA -- drivers must navigate terminals
- **Highway zones:** High speed but requires detours to reach pickup points

**If zone has strong signal** (high variance across zones), we encode `PULocationID` as a feature.

**Encoding choice:** With ~265 unique zones:
- One-Hot: 265 sparse binary columns -- dimensionality problems, multicollinearity
- **Target Encoding:** Replace each zone with its (smoothed) mean ETA -- single column,
  captures ordinal relationship, no dimensionality explosion

Target encoding is the standard approach for high-cardinality categoricals in ML.


In [ ]:
# Cell 20: Zone-level ETA analysis
MIN_TRIPS = 100  # minimum trips per zone for reliable estimate

zone_eta = (
    df.groupby('PULocationID')['eta_seconds']
    .agg(['mean','median','count'])
    .rename(columns={'mean':'mean_eta','median':'median_eta','count':'n_trips'})
    .query(f'n_trips >= {MIN_TRIPS}')
    .sort_values('mean_eta', ascending=False)
)

print(f'Zones with >= {MIN_TRIPS} trips: {len(zone_eta)}')
print(f'Mean ETA std across zones:       {zone_eta["mean_eta"].std():.1f}s')
print(f'Zone ETA range: [{zone_eta["mean_eta"].min():.0f}s, {zone_eta["mean_eta"].max():.0f}s]')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_slow = zone_eta.head(10).reset_index()
sns.barplot(data=top_slow, x='mean_eta', y='PULocationID',
            orient='h', palette='Reds_r', ax=axes[0])
axes[0].set_title('Top 10 Slowest Pickup Zones')
axes[0].set_xlabel('Mean ETA (seconds)')

top_fast = zone_eta.tail(10).reset_index()
sns.barplot(data=top_fast, x='mean_eta', y='PULocationID',
            orient='h', palette='Greens_r', ax=axes[1])
axes[1].set_title('Top 10 Fastest Pickup Zones')
axes[1].set_xlabel('Mean ETA (seconds)')

plt.suptitle('ETA by Pickup Zone -- Signal for Target Encoding', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_eta_by_zone.png', dpi=120, bbox_inches='tight')
plt.show()
print('High variance across zones confirms: PULocationID and DOLocationID -> Target Encode.')


## 2.21  EDA Summary -- Feature Engineering Decision Log

Every analysis above translates to a concrete decision:

| Analysis | Finding | Decision |
|---|---|---|
| Target distribution | `eta_seconds` is right-skewed | Apply `log1p` transform to target |
| IQR outliers | Outliers in both `eta_seconds` and `trip_miles` | Cap at IQR bounds in preprocessing |
| Uber vs Lyft test | Statistically different ETAs (p < 0.05) | Include `hvfhs_license_num` as feature |
| Rush hour | Clear ETA elevation at 7-9am and 5-7pm | Create `pickup_hour` + `is_rush_hour` |
| Weekend test | Significant distribution difference | Include `is_weekend` as binary feature |
| Zone analysis | High variance across 260+ zones | Target-encode `PULocationID`, `DOLocationID` |
| Spearman corr | `trip_time` highly correlated with ETA | CANNOT USE -- post-trip data leakage |
| Spearman corr | `trip_miles` correlated with ETA | Include (with caution -- verify availability at request time) |

**Data Leakage Warning -- Columns That MUST NOT Be Features:**
- `trip_time` -- total trip duration (only known when trip ends)
- `dropoff_datetime` -- end of trip (future information)
- `base_passenger_fare`, `driver_pay`, `tips`, `tolls` -- post-trip financial data
- `on_scene_datetime` -- this IS our target, cannot also be a feature

---

## Key Interview Questions -- Notebook 02

**Q: Why Mann-Whitney U and not a t-test?**  
A: The t-test assumes normality within each group. Our `eta_seconds` is heavily right-skewed
-- the normality assumption is violated. Mann-Whitney U is a non-parametric rank-based test
that makes no distributional assumptions and has nearly equal power for large samples.

**Q: What is the CLT and why does it matter for ML?**  
A: The Central Limit Theorem guarantees that sampling distributions of means approach
normal regardless of the population distribution. This justifies using standard
statistical tests on our data even though the raw ETA distribution is skewed.

**Q: Why Spearman over Pearson here?**  
A: Pearson requires bivariate normality and measures only linear relationships.
Our data is skewed and relationships are likely monotonic but non-linear.
Spearman's rank correlation is robust to both conditions.

**Q: What is data leakage and which columns are leaking?**  
A: Data leakage is when information unavailable at prediction time is used as a feature.
Leaky columns: `trip_time`, `dropoff_datetime`, `base_passenger_fare`, `driver_pay`.
All are only known after the trip ends -- exactly when we do NOT need to predict ETA.

**Q: Why target-encode zones instead of one-hot encoding?**  
A: 265 unique zones would create 265 sparse binary columns. This causes dimensionality
problems, multicollinearity, and slow training. Target encoding replaces each zone with
its smoothed mean ETA -- a single informative numeric column. Industry standard for
high-cardinality categoricals.

**Q: What does skewness tell you about log-transforming the target?**  
A: Skewness > 1 means a long right tail -- extreme values disproportionately influence
MSE/RMSE loss. Log-transform compresses the tail, balancing the influence of all data
points during training. After prediction, inverse-transform with `expm1()` to get seconds.
